# SIMKD + TGSC on CIFAR-100

Enable a Kaggle GPU and Internet for clone/install/download. Choose downloaded or attached teacher/CIFAR sources in the settings cell. Training notebooks also need a prepared TGSC crop bank. Use the same code revision and teacher weights for preparation and training.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Edit this cell. Paths below are examples; use your attached artifact locations.
REPO_URL = "https://github.com/ahmad903501/tgsc-mdistiller.git"
PRIVATE_REPO = False  # Public repository; no GitHub token is needed.
REVISION = "499a60f"  # Match the revision used by the preparation notebook.
REPO_DIR = Path("/kaggle/working/mdistiller")
CONFIG_NAME = "simkd_resnet32x4_to_resnet8x4.json"
GPU = 0  # One GPU per process; other jobs can use another GPU.
SEED = 27
CIFAR_SOURCE = "download"  # "download" or "kaggle"
DATA_ROOT = Path("/kaggle/working/data")  # Parent of cifar-100-python/
TEACHER_SOURCE = "kaggle"  # "download" or "kaggle"
TEACHER_DOWNLOAD_ROOT = Path("/kaggle/working/teacher_weights")
TEACHER_CHECKPOINT = Path("/kaggle/input/tgsc-teacher-resnet32x4/cifar_teachers/resnet32x4_vanilla/ckpt_epoch_240.pth")
CACHE_ROOT = Path("/kaggle/input/tgsc-crop-bank")  # Parent of bank hash folders
CACHE_WRITE_ROOT = Path("/kaggle/working/tgsc_cache")
OUTPUT_DIR = Path("/kaggle/working/tgsc_runs/simkd_res32x4_res8x4_seed27")
RESUME = None  # E.g. Path('/kaggle/input/previous-run/latest.pt')


In [ ]:
if not REPO_DIR.exists():
    if "YOUR_ACCOUNT" in REPO_URL:
        raise ValueError("Set REPO_URL to your pushed repository")
    def clone_repository():
        if not PRIVATE_REPO:
            subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
            return
        import os
        import tempfile
        from kaggle_secrets import UserSecretsClient
        # Keep the token out of URLs, notebook output, and files.
        clone_env = dict(os.environ, TGSC_GITHUB_TOKEN=UserSecretsClient().get_secret("GITHUB_TOKEN"),
                         GIT_TERMINAL_PROMPT="0")
        with tempfile.TemporaryDirectory() as auth_directory:
            askpass = Path(auth_directory) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['TGSC_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            clone_env["GIT_ASKPASS"] = str(askpass)
            subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], env=clone_env, check=True)
    clone_repository()
    subprocess.run(["git", "checkout", REVISION], cwd=REPO_DIR, check=True)
else:
    wanted = subprocess.check_output(["git", "rev-parse", REVISION], cwd=REPO_DIR, text=True).strip()
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    if wanted != actual:
        raise RuntimeError("Existing checkout differs from REVISION; select the intended checkout before running")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-tgsc.txt")], check=True)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())


## Choose teacher and CIFAR-100 sources

Set `TEACHER_SOURCE="download"` to download the official combined archive and extract **only the teacher selected by CONFIG_NAME**. The official release has no individual teacher assets; the archive download is about 348 MiB. Set `TEACHER_SOURCE="kaggle"` to use an attached single-model Dataset at `TEACHER_CHECKPOINT`, without downloading teacher weights. Training preflight verifies that it loads into the configured architecture; cache identity also checks the checkpoint hash.

`CIFAR_SOURCE="download"` uses torchvision's original CIFAR-100 downloader, exactly as TGSC does, with writable `DATA_ROOT=/kaggle/working/data`. Use `CIFAR_SOURCE="kaggle"` for an attached copy and set `DATA_ROOT` to the **parent of `cifar-100-python/`**. A Dataset containing only CSV/PNG files is not the torchvision layout. Keep original sample order and labels so the prepared cache stays compatible.

For later sessions, preserve the selected checkpoint folder plus `teacher_manifest.json` as a teacher Dataset, and `data/cifar-100-python/` as a CIFAR Dataset. Do not upload the combined teacher tar: only the selected `.pth` is needed. Upload/attach these through Kaggle's Dataset UI; this notebook does not publish them automatically.


In [ ]:
if TEACHER_SOURCE not in {"download", "kaggle"}:
    raise ValueError('TEACHER_SOURCE must be "download" or "kaggle"')
if CIFAR_SOURCE not in {"download", "kaggle"}:
    raise ValueError('CIFAR_SOURCE must be "download" or "kaggle"')
DOWNLOAD = CIFAR_SOURCE == "download"
if DOWNLOAD and DATA_ROOT.as_posix().startswith("/kaggle/input/"):
    raise ValueError("CIFAR download needs a writable DATA_ROOT, e.g. /kaggle/working/data")
if not DOWNLOAD and not (DATA_ROOT / "cifar-100-python").is_dir():
    raise FileNotFoundError("DATA_ROOT must contain cifar-100-python/ from the original CIFAR archive")
if TEACHER_SOURCE == "download":
    subprocess.run([
        sys.executable, str(REPO_DIR / "tools" / "download_teachers.py"),
        "--config", str(REPO_DIR / "configs" / "tgsc" / "cifar100" / CONFIG_NAME),
        "--output-dir", str(TEACHER_DOWNLOAD_ROOT),
    ], cwd=REPO_DIR, check=True)
    import json
    manifest = json.loads((TEACHER_DOWNLOAD_ROOT / "teacher_manifest.json").read_text())
    TEACHER_CHECKPOINT = TEACHER_DOWNLOAD_ROOT / manifest["relative_checkpoint"]
if not TEACHER_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Teacher checkpoint does not exist: {TEACHER_CHECKPOINT}")
print("Teacher checkpoint:", TEACHER_CHECKPOINT)
print("CIFAR-100:", CIFAR_SOURCE, DATA_ROOT)


In [ ]:
CONFIG = REPO_DIR / "configs" / "tgsc" / "cifar100" / CONFIG_NAME
RUNNER = REPO_DIR / "tools" / "run_tgsc.py"

def run_tgsc(command, *extra):
    args = [sys.executable, "-u", str(RUNNER), command,
            "--config", str(CONFIG), "--gpu", str(GPU), "--seed", str(SEED),
            "--data-root", str(DATA_ROOT), "--cache-root", str(CACHE_ROOT)]
    if command != "evaluate":
        args += ["--teacher-checkpoint", str(TEACHER_CHECKPOINT)]
    if DOWNLOAD:
        args += ["--download"]
    args += [str(value) for value in extra]
    log_dir = CACHE_WRITE_ROOT if command == "prepare" else OUTPUT_DIR
    log_dir.mkdir(parents=True, exist_ok=True)
    with (log_dir / (command + ".log")).open("a", encoding="utf-8") as log:
        process = subprocess.Popen(args, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, args)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise


## Check before training

A compatible prepared crop bank is required. Use the preparation notebook if the runner reports missing cache files.


In [ ]:
run_tgsc("check")


## Train or resume

The runner saves each completed epoch. Training uses the existing TGSC curriculum and default augmentation. No new baseline training is performed.


In [ ]:
extra = ["--output-dir", OUTPUT_DIR]
if RESUME is not None:
    extra += ["--resume", RESUME]
run_tgsc("train", *extra)


## Evaluate the exported model

SimKD exports its projector and reused teacher head as well as the backbone. Evaluation does not require teacher-backbone weights. Save notebook outputs to preserve checkpoints/logs for a later session.


In [ ]:
run_tgsc("evaluate", "--export", OUTPUT_DIR / "inference_best.pt")
